In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks, select_period

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"

endpoint_columns = [
    "sender_bank",
    "sender_account",
    "receiver_bank",
    "receiver_account",
]

In [2]:
history_parts = []
row_offset = 0

chunks = iter_transaction_chunks(
    csv_path,
    usecols=["timestamp", *endpoint_columns],
)

for chunk in chunks:
    # Assign source positions before filtering, exactly as in Lab 2.
    chunk["source_row"] = range(
        row_offset,
        row_offset + len(chunk),
    )
    row_offset += len(chunk)

    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    selected = select_period(
        chunk,
        start=pd.Timestamp("2022-09-01"),
        end=pd.Timestamp("2022-09-09"),
    )

    if not selected.empty:
        for column in endpoint_columns:
            selected[column] = selected[column].astype(
                "string[pyarrow]"
            )

        history_parts.append(selected)

history = (
    pd.concat(history_parts, ignore_index=True)
    .sort_values(["timestamp", "source_row"])
    .reset_index(drop=True)
)

# Release the intermediate list after combining it.
history_parts.clear()

print(f"History transactions: {len(history):,}")
print(
    "History memory:",
    round(history.memory_usage(deep=True).sum() / 1024**2, 1),
    "MiB",
)
display(history.head())

History transactions: 4,214,445
History memory: 313.6 MiB


,timestamp,sender_bank,sender_account,receiver_bank,receiver_account,source_row
0,2022-09-01,03209,8000F4670,03209,8000F4670,2
1,2022-09-01,01420,8005DFEB0,01420,8005DFEB0,17
2,2022-09-01,010,8000F6850,010,8000F6850,158
3,2022-09-01,012,8001167D0,012,8001167D0,218
4,2022-09-01,001,8001364A0,001,8001364A0,281


In [5]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import attach_account_endpoints

In [6]:
from src.data import attach_account_endpoints

train_base = pd.read_parquet(
    project_root / "data/processed/lab02_train.parquet"
)
validation_base = pd.read_parquet(
    project_root / "data/processed/lab02_validation.parquet"
)

train = attach_account_endpoints(train_base, history)
validation = attach_account_endpoints(validation_base, history)

assert len(history) == 4_214_445
assert history["source_row"].is_unique

assert len(train) == 1_379_551
assert len(validation) == 965_524

assert int(train["is_laundering"].sum()) == 1_800
assert int(validation["is_laundering"].sum()) == 1_036

display(
    train[
        ["source_row", "timestamp", *endpoint_columns]
    ].head()
)

,source_row,timestamp,sender_bank,sender_account,receiver_bank,receiver_account
0,1870235,2022-09-03,01292,80015F700,010,80015F8E0
1,1870247,2022-09-03,012,8001A8CC0,0111425,80B7FE8F0
2,1870250,2022-09-03,070,100428660,01362,8001C4B70
3,1870255,2022-09-03,012,8001CB7C0,0011642,804DE86F0
4,1870258,2022-09-03,001,8001C9040,010,8001CC420


### Historical-feature setup

We loaded account endpoints and timestamps for September 1–8.
September 1–2 supplies earlier context for training transactions.

We restored endpoints using source-row identifiers, checking one-to-one
matching, timestamp agreement, and transaction order.

Historical features will use the preceding 24 hours, excluding the
current timestamp. Earlier validation activity is available to later
transactions, but laundering labels are excluded from history features.

In [ ]:
from time import perf_counter

from src.history import HISTORY_FEATURES, make_history_features

# Targets are the transactions we will train on or evaluate.
# History also includes September 1–2 as earlier context.
targets = select_period(
    history,
    start=pd.Timestamp("2022-09-03"),
    end=pd.Timestamp("2022-09-09"),
)

started = perf_counter()

history_features = make_history_features(history, targets)

print(f"Calculation took {perf_counter() - started:.1f} seconds.")
print(f"Feature rows: {len(history_features):,}")
display(history_features.head())

In [ ]:
train_with_history = train.merge(
    history_features,
    on="source_row",
    how="left",
    sort=False,
    validate="one_to_one",
)

validation_with_history = validation.merge(
    history_features,
    on="source_row",
    how="left",
    sort=False,
    validate="one_to_one",
)

for original, enriched in [
    (train, train_with_history),
    (validation, validation_with_history),
]:
    assert enriched["source_row"].equals(original["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()
    assert enriched[HISTORY_FEATURES].ge(0).all().all()

display(
    train_with_history[HISTORY_FEATURES]
    .describe(percentiles=[0.5, 0.9, 0.99])
    .round(1)
)